# 📚 RAG From Scratch — Insurellm Knowledge Worker

*A reference notebook: building a Retrieval-Augmented Generation (RAG) pipeline **without** a framework like LangChain, so every moving part is visible.*

---

## The problem

An LLM knows nothing about the private documents of our (fictional) insurance-tech company **Insurellm** — its products, employees, and contracts. Fine-tuning a model on them is expensive and goes stale the moment a document changes.

## The solution: RAG

Don't teach the model the documents — **look up the relevant snippets at question time and paste them into the prompt**. The model then answers *grounded* in real company text instead of guessing.

RAG has two phases:

```
PHASE 1 — INGESTION (run once, offline)
  markdown files ──▶ LLM-powered chunking ──▶ embeddings ──▶ Chroma vector DB

PHASE 2 — ANSWERING (every question)
  question ──▶ rewrite query ──▶ embed it ──▶ vector search (top-K)
           ──▶ LLM rerank ──▶ build prompt with the context ──▶ LLM answers
```

## Tech stack

| Library | Role |
|---|---|
| `litellm` | one `completion()` API for chat calls (chunking, reranking, rewriting, answering) |
| `openai` | embedding calls (`text-embedding-3-large`, 3072 dimensions) |
| `chromadb` | local on-disk **vector database** (stores vectors + text + metadata) |
| `pydantic` | **structured outputs** — forces LLM replies to match a JSON schema |
| `sklearn` + `plotly` | t-SNE dimensionality reduction + interactive plots to inspect the vector space |
| `tqdm`, `dotenv` | progress bars; loading API keys from `.env` |

## Key vocabulary

- **Embedding** — a list of numbers (here 3072 of them) representing the *meaning* of a text. Similar meaning ⇒ nearby vectors.
- **Vector store** — a database that finds the stored vectors nearest to a query vector, fast.
- **Chunk** — a bite-sized piece of a document. We search and prompt with chunks, not whole documents.
- **Top-K retrieval** — fetch the K chunks nearest to the question's embedding.
- **Reranking** — a second pass where an LLM re-orders the retrieved chunks by true relevance.
- **Grounding** — making the model answer from supplied text, which greatly reduces hallucination.


## ⚙️ Setup — imports & configuration

Two different clients are used on purpose:

- **`litellm.completion`** for every *chat* call — one uniform API, and switching providers is just a string change (`'openai/...'` → `'anthropic/...'`).
- **`OpenAI()`** directly for *embeddings*, which LiteLLM isn't used for here.

The constants below are the knobs of the whole system — model names, where the vector DB lives on disk, and the target chunk size.

> **Summary:** load secrets from `.env`, pick models, point at the `knowledge-base/` folder, open the clients.


In [42]:
# ── Imports ──────────────────────────────────────────────────────────────────
from pathlib import Path               # object-oriented filesystem paths (walks the knowledge base)
from openai import OpenAI              # official OpenAI client — used ONLY for embeddings here
from dotenv import load_dotenv         # loads API keys from the .env file into environment variables
from pydantic import BaseModel, Field  # data validation + "structured outputs" (forces LLM replies into a schema)
from chromadb import PersistentClient  # Chroma: a local, file-based vector database
from tqdm import tqdm                  # progress bars for slow loops
from litellm import completion         # one chat-completions API that works across many LLM providers
import numpy as np                     # numeric arrays (holds the embedding vectors)
from sklearn.manifold import TSNE      # t-SNE: squashes 3072-D vectors down to 2-D/3-D for plotting
import plotly.graph_objects as go      # interactive scatter plots to "see" the vector space

In [43]:
load_dotenv(override=True)  # read OPENAI_API_KEY etc. from .env; override any stale values already set

# ── Configuration ────────────────────────────────────────────────────────────
MODEL = 'openai/gpt-6-luna'       # chat model for chunking, reranking, query rewriting AND answering
                                      # (LiteLLM naming: 'provider/model')

DB_NAME = 'preprocessed_db'           # folder on disk where Chroma persists the vector database
collection_name = 'docs'              # a Chroma "collection" is like a table of (vector, text, metadata) rows
embedding_model = 'text-embedding-3-large'   # OpenAI embedding model: text -> 3072-dimensional vector
KNOWLEDGE_BASE_PATH = Path('knowledge-base') # root folder of the markdown documents to ingest
AVERAGE_CHUNK_SIZE = 500              # target chunk size in CHARACTERS — used to suggest a chunk count to the LLM

openai = OpenAI()  # client instance used for the embedding calls

## 🧱 Data models — Pydantic schemas as contracts

Three tiny classes do a lot of work:

- **`Result`** — the universal "retrieved chunk" container (LangChain calls this a `Document`): the text plus metadata about where it came from.
- **`Chunk`** — the schema the LLM must follow when it splits a document. Passed as `response_format=`, it makes the API **guarantee** a JSON reply matching this shape. The `Field(description=...)` strings are *read by the model* — they are part of the prompt!
- **`Chunks`** — a wrapper so the model can return a *list* of chunks in one JSON object.

💡 The clever bit is `Chunk.as_result()`: the chunk's **headline + summary + original text** are concatenated into one string *before embedding*. A user's question often matches the LLM-written summary even when it shares no words with the raw text — this measurably improves retrieval.

> **Summary:** define the shapes data must have, and let Pydantic validate every LLM reply against them.


In [44]:
# Result is this notebook's universal "retrieved chunk" container
# (equivalent to LangChain's Document class):
#   - page_content: the text that gets embedded, searched, and pasted into prompts
#   - metadata:     where the text came from (source file path + document type)
class Result(BaseModel):
    page_content: str
    metadata: dict

In [45]:
# The schema the LLM must follow when chunking a document ("structured outputs").
# Passing this model as response_format= forces the LLM to return valid JSON that
# matches these fields — and the Field descriptions are READ BY THE MODEL as
# instructions, so write them carefully.
class Chunk(BaseModel):
    headline: str = Field(description='A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query')
    summary: str = Field(description='A few sentences summarizing the content of this chunk to answer common questions')
    original_text: str = Field(description='The original text of this chunk from the provided document, exactly as is, not changed in any way')

    def as_result(self, document):
        # Fold headline + summary + original text into ONE searchable string.
        # Embedding the LLM-written headline/summary alongside the raw text is a
        # cheap trick that improves retrieval a lot: a user's query often matches
        # the summary even when it shares no words with the original text.
        metadata = {'source': document['source'], 'type': document['type']}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" +self.original_text, metadata=metadata)

# Wrapper: the LLM returns a single JSON object containing the list of chunks
class Chunks(BaseModel):
    chunks: list[Chunk]

## 🏗️ Phase 1 — Ingestion

Three steps, run once (or whenever the knowledge base changes):

1. **Load** — read every markdown file from `knowledge-base/` into memory.
2. **Chunk** — have an LLM split each document into overlapping, labelled chunks.
3. **Embed & store** — turn each chunk into a vector and save it all in Chroma.


## Three Steps
### Step 1

### Step 1 — Load the documents

Walk `knowledge-base/`, where each sub-folder (`company`, `contracts`, `employees`, `products`) is a document *type*. Each file becomes a dict of `{type, source, text}` — the `type` and `source` ride along as metadata forever, so final answers can cite which file they came from.

> **Summary:** 76 markdown files → 76 document dicts.


In [46]:
def fetch_documents():
    """ A homemade version of the LangChain DirectoryLoader """
    documents = []

    # knowledge-base/ is organized as one sub-folder per document type:
    #   knowledge-base/company, /contracts, /employees, /products
    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name   # the folder name becomes the 'type' metadata
        for file in folder.rglob("*.md"):   # recursively find every markdown file
            with open(file, 'r', encoding='utf-8') as f:
                documents.append({'type': doc_type, 
                                  'source': file.as_posix(),  # file path kept so answers can cite their source
                                  'text': f.read()})
    print(f"Loaded {len(documents)} documents")
    return documents

In [47]:
# Run Step 1 — expect: 'Loaded 76 documents'
documents = fetch_documents()

Loaded 76 documents


### Step 2

### Step 2 — LLM-powered chunking

**Why chunk at all?** Embedding a whole document into one vector dilutes its meaning (one vector can't represent 10 different topics), and whole documents would blow the prompt budget. Small chunks give precise retrieval.

**Why use an LLM to chunk?** The naive approach (split every N characters) slices sentences and tables in half. Here the *LLM reads the document and chooses sensible boundaries*, keeps ~25% overlap between neighbouring chunks (so a fact near a boundary is retrievable from either side), and writes a **headline + summary** for each chunk that supercharges retrieval.

**The trade-off:** one chat-model call per document — about 18 minutes and real token cost for 76 docs — versus a microsecond string split. For a high-value knowledge base, worth it.

> **Summary:** 76 documents → one LLM call each → **593** overlapping, labelled chunks.


In [48]:
# Build the chunking prompt for ONE document.
# Instead of splitting text with a dumb character counter, we hand the WHOLE
# document to an LLM and let it choose sensible boundaries, with ~25% overlap
# so facts near a boundary appear in both neighbouring chunks.
# `how_many` is just a size hint: doc length / AVERAGE_CHUNK_SIZE characters.
def make_prompt(document):
    how_many = (len(document['text']) // AVERAGE_CHUNK_SIZE) + 1
    return f""" 
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm. The document is of type:
{document['type']}. The document has been retrieved from: {document['source']}. 

A chatbot will use these chunks to answer question about the company. You should divide up the
document as you see fit, being sure that the entire document is returned in the chunks - don't 
leave anything out. This document should probably be split into {how_many} chunks, but you can hve 
more or less as appropriate. There should be overlap between the chunks as appropriate; typically
about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk. Together
your chunks should represent the entire document with overlap.

Here is the document:
{document['text']}

Respond with the chunks.
"""

In [49]:
# Sanity check: what does the prompt actually look like for the first document?
print(make_prompt(documents[0]))

 
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm. The document is of type:
products. The document has been retrieved from: knowledge-base/products/Rellm.md. 

A chatbot will use these chunks to answer question about the company. You should divide up the
document as you see fit, being sure that the entire document is returned in the chunks - don't 
leave anything out. This document should probably be split into 8 chunks, but you can hve 
more or less as appropriate. There should be overlap between the chunks as appropriate; typically
about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk. Together
your chunks should represent the entire document with overlap.

Here is the document:
# Product Summary

# Rellm: AI-Powered Enterpri

In [50]:
# Wrap the chunking prompt in the chat-messages format every LLM API expects.
# A single 'user' message is enough here — no system prompt needed for this job.
def make_messages(document):
    return [
        {
            'role': 'user',
            'content': make_prompt(document)
        },
    ]

In [51]:
# Sanity check: the same prompt wrapped as a messages list
make_messages(documents[0])

[{'role': 'user',
  'content': " \nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm. The document is of type:\nproducts. The document has been retrieved from: knowledge-base/products/Rellm.md. \n\nA chatbot will use these chunks to answer question about the company. You should divide up the\ndocument as you see fit, being sure that the entire document is returned in the chunks - don't \nleave anything out. This document should probably be split into 8 chunks, but you can hve \nmore or less as appropriate. There should be overlap between the chunks as appropriate; typically\nabout 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk. Together\nyour chunks should represent the entire document with overlap.\n\nHere is the document:\

In [52]:
# Send ONE document to the LLM and get back a list of validated Result objects
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, 
                          extra_body={'reasoning_effort': 'none'},  # no "thinking" needed — faster + cheaper
                          messages=messages,
                          response_format=Chunks)   # structured outputs: reply MUST be JSON matching Chunks
    reply = response.choices[0].message.content     # the raw JSON string
    doc_as_chunks = Chunks.model_validate_json(reply).chunks   # parse + validate into Chunk objects
    return [chunk.as_result(document) for chunk in doc_as_chunks]  # attach metadata, convert to Result

In [53]:
# Sanity check: chunk ONE document and inspect the Result objects
process_document(documents[0])

[Result(page_content='Rellm Product Summary\n\nRellm is Insurellm’s AI-powered enterprise reinsurance product. It aims to transform reinsurance operations through AI, robust analytics, integrations, improved risk management, and more agile decision-making.\n\n# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.', metadata={'source': 'knowledge-base/products/Rellm.md', 'type': 'products'}),
 Result(page_content="AI Analytics and Integrations\

In [54]:
# Chunk every document — one LLM call per document, so this is the SLOW step
# (~14 s per doc, ~18 minutes for all 76 docs). tqdm shows a live progress bar.
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [55]:
# ⏳ The big one: chunk all 76 documents (~18 minutes, one LLM call each)
chunks = create_chunks(documents)

100%|██████████| 76/76 [18:38<00:00, 14.72s/it]


In [56]:
# 76 documents became 593 overlapping chunks
print(len(chunks))

588


### Step 3

### Step 3 — Embed & store in Chroma

Each chunk's text goes to `text-embedding-3-large` and comes back as a **3072-dimensional vector** that captures its meaning. All vectors, texts, and metadata are stored together in a Chroma **collection**, persisted to the `preprocessed_db/` folder on disk — so you only pay the ingestion cost once; later sessions just reopen the folder.

The function deletes any existing collection first, so it is safe to re-run (idempotent).

> **Summary:** 593 chunks → 593 vectors in a searchable, on-disk database.


In [57]:
EMBED_BATCH_SIZE = 500   # texts per embedding request (OpenAI caps requests at 2048 inputs / token limits)

def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)   # open (or create) the on-disk database

    # Idempotency: wipe any existing collection so re-running starts fresh
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    # Embed the chunks in batches. Each text becomes a 3072-dim vector; texts
    # with similar MEANING get vectors that are close together.
    texts = [chunk.page_content for chunk in chunks]
    vectors = []
    for i in range(0, len(texts), EMBED_BATCH_SIZE):
        batch = texts[i:i + EMBED_BATCH_SIZE]
        emb = openai.embeddings.create(model=embedding_model, input=batch).data
        vectors.extend(e.embedding for e in emb)

    collection = chroma.get_or_create_collection(collection_name)

    # Chroma rows need unique string ids — the list position works fine here
    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    # Store vector + original text + metadata side by side, so a vector search
    # can hand back the human-readable chunk and its source, not just numbers
    collection.add(ids=ids,
                   embeddings=vectors,
                   documents=texts,
                   metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [58]:
# Embed all 593 chunks and store them — expect: 'Vectorstore created with 593 documents'
create_embeddings(chunks)

Vectorstore created with 588 documents


### 👀 Visualizing the vector space (optional sanity check)

You can't picture 3072 dimensions, but **t-SNE** can squash the vectors into 2-D/3-D while trying to keep close neighbours close. Colouring each point by document type should show loose clusters — products near products, employees near employees. If the plot were one undifferentiated blob, the embeddings (and retrieval) would be suspect.

> **Summary:** a visual confidence check that "similar content ⇒ similar vectors" actually happened.


In [59]:
# ── Load everything back out of Chroma for visualization ─────────────────────
# `chunk_texts` (not `documents`) so we don't shadow the document dicts from Step 1
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings',
                                 'documents',
                                 'metadatas'])
vectors = np.array(result['embeddings'])      # shape: (593, 3072)
chunk_texts = result['documents']             # the chunk texts
metadatas = result['metadatas']               # their source/type metadata
doc_types = [metadata['type'] for metadata in metadatas]
# one color per document type, for the scatter plots below
colors = [['blue',
           'green',
           'red',
           'orange'][['products',
                     'employees',
                     'contracts',
                     'company'].index(t)] for t in doc_types] 


In [60]:
# t-SNE squeezes the 3072-D vectors into 2-D while trying to keep neighbours close.
# This is a sanity check of the vector store: chunks of the same type (colour)
# should form loose clusters — meaning 'similar content got similar vectors'.
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5,
                color=colors,
                opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, chunk_texts)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
                  scene=dict(xaxis_title='x',
                             yaxis_title='y'),
                             width=800,
                             height=600,
                             margin=dict(r=20,
                                         b=10,
                                         l=10,
                                         t=40))
fig.show()

In [61]:
# Same idea in 3-D. (Note: the indentation of text=/hoverinfo= is misleading —
# they are arguments to Scatter3d, not to marker=dict(...). The code is valid.)
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5,
                color=colors,
                opacity=0.8),
                text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, chunk_texts)],
                hoverinfo='text'
)])

fig.update_layout(title='3D Chroma Vector Store Visualization',
                  scene=dict(xaxis_title='x',
                             yaxis_title='y',
                             zaxis_title='z'),
                             width=900,
                             height=700,
                             margin=dict(r=10,
                                         b=10,
                                         l=10,
                                         t=40))
fig.show()

## Build Advanced RAG

**Basic RAG** would stop at: embed the question → take the top-K chunks → answer.

This section adds the two upgrades that make it **advanced RAG**:

1. **Re-ranking** — vector search is approximate, so *retrieve wide* (10–20 chunks), then let an LLM that actually reads them put the best ones first.
2. **Query rewriting** (further below) — rewrite the user's conversational question into a sharp search query before retrieving.


In [62]:
# Structured-output schema for the reranker: the LLM replies with just a list
# of chunk id numbers, best first — e.g. [3, 1, 7, ...]
class RankOrder(BaseModel):
    order: list[int] = Field(
        description='The order of relevance of chunks, from most relevant to least relevant, by chunk id number'
    )

In [63]:
# ── Re-ranking ──────────────────────────────────────────────────────────────
# Vector search is fast but approximate: it compares the question to each chunk
# in isolation. A second pass where an LLM READS the question and all retrieved
# chunks together, then re-orders them by true relevance, is a cheap upgrade.
# Chunks are numbered 1..N in the prompt; the model replies (via the RankOrder
# schema) with ids in relevance order, and `chunks[i - 1]` maps back to 0-based.
def rerank(question, chunks):
    system_prompt = """ 
You are a document re-ranker. You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to 
improve on that. You must rank order the provided chunks by relevance to the question, with the most relevant chunk first. Reply only 
with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""

    user_prompt = f"""The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most
    relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n
    """
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n" 
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {
            'role': 'system',
            'content': system_prompt
        },
        {
            'role': 'user',
            'content': user_prompt
        },
    ]
    response = completion(model=MODEL,
                           extra_body={'reasoning_effort': 'none'},
                          messages=messages,
                          response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    # Guard against model mistakes: drop invalid or duplicate ids, then append
    # any ids the model forgot, so no chunk is ever lost and nothing can crash
    seen = set()
    valid = []
    for i in order:
        if 1 <= i <= len(chunks) and i not in seen:
            valid.append(i)
            seen.add(i)
    valid += [i for i in range(1, len(chunks) + 1) if i not in seen]
    return [chunks[i - 1] for i in valid]

In [64]:
RETRIEVAL_K = 10   # default number of nearest-neighbour chunks to retrieve

def fetch_context_unranked(question, k=RETRIEVAL_K):
    # 1) Embed the question with the SAME model used for the chunks
    #    (vectors from different models live in different spaces — distances
    #     between them are meaningless)
    query = openai.embeddings.create(model=embedding_model,
                                     input=[question]).data[0].embedding
    # 2) Ask Chroma for the k stored vectors closest to the question's vector
    results = collection.query(query_embeddings=[query], n_results=k)
    # 3) Chroma returns parallel lists ([0] = results for the first/only query);
    #    zip them back together into Result objects
    chunks = []
    for result in zip(results['documents'][0],
                       results['metadatas'][0]):
        chunks.append(Result(page_content=result[0], 
                             metadata=result[1]))
    return chunks   # must sit OUTSIDE the loop — inside, it returns after one chunk!

### 🧪 Testing retrieval + reranking

Two experiments prove the reranker earns its keep:

- **Test 1 (IIOTY award):** retrieve 10 chunks, print them unranked, then reranked — watch the order change.
- **Test 2 (Manchester University):** retrieve **20** chunks (via the `k=20` argument) and print *where* the chunk mentioning Manchester lands. Plain vector search tends to bury it (a question about a *person's university* isn't lexically close to an *HR record*); after reranking it should sit at **index 0**.


In [65]:
# Test 1 — IIOTY = the 'Insurance Innovator of the Year' award in the HR records
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [66]:
# What came back, in raw vector-distance order (nearest first)
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

HR Feedback and...
Performance Rev...
Annual Performa...
Enterprise Suit...
Michael O’Brien...
Performance His...
Daniel Park — D...
Training, Award...
Insurellm Caree...
Maxine Thompson...


In [67]:
# Ask the LLM to re-order those chunks by true relevance (prints the new order)
reranked = rerank(question, chunks)

[10, 1, 8, 3, 7, 2, 5, 6, 9, 4]


In [68]:
# The same chunks, now in LLM-judged relevance order
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

Maxine Thompson...
HR Feedback and...
Training, Award...
Annual Performa...
Daniel Park — D...
Performance Rev...
Michael O’Brien...
Performance His...
Insurellm Caree...
Enterprise Suit...


In [69]:
# Test 2 (harder) — retrieve 20 chunks, then print the INDEX where the
# 'Manchester' chunk landed. Plain vector search often buries it deep in the list.
question = 'Who went to Manchester University?'
chunks = fetch_context_unranked(question, k=20)   # widen the net for this test only (no global mutation)
for index, c in enumerate(chunks):
    if 'manchester' in c.page_content.lower():
        print(index)


3


In [70]:
# ...reranking should pull the Manchester chunk to (or near) the front
reranked = rerank(question, chunks)

[4, 1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


In [71]:
# Its position after reranking — expect 0 (the top)
for index, c in enumerate(reranked):
    if 'manchester' in c.page_content.lower():
        print(index)

0


In [72]:
# Confirm the top chunk really answers the question
reranked[0].page_content

'Jessica Liu: Education, Skills, and Work Style\n\nJessica Liu has a BS in Computer Science from the University of Manchester. She is proficient in React, TypeScript, HTML/CSS, and Jest, and is learning Next.js and GraphQL. She completed an Advanced React Patterns course in 2023, contributes to open-source projects, prefers remote work, and communicates well in writing while actively participating in team standups and planning.\n\n## Other HR Notes\n- **Education:** BS in Computer Science from University of Manchester\n- **Skills:** Proficient in React, TypeScript, HTML/CSS, Jest for testing. Learning Next.js and GraphQL.\n- **Professional Development:** Completed Advanced React Patterns course (2023). Actively contributes to open-source projects.\n- **Work Style:** Prefers remote work. Strong written communicator. Participates actively in team standups and planning sessions.\n- **Feedback:** Reliable developer with good attention to UI details. Improving technical decision-making skil

In [73]:
# The full retrieval pipeline: vector search casts a wide net,
# then the LLM reranker sorts the catch ("retrieve wide, rerank narrow")
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

## 💬 Generation — answering with the retrieved context

The last mile: a system prompt with a `{context}` placeholder that gets filled with the retrieved chunks (each labelled `Extract from <source file>` so answers can cite sources). The message list is `[system + context] + history + [question]`, which supports multi-turn conversation.

Telling the model *"If you don't know the answer, say so"* is the anti-hallucination safety net — without it, LLMs confidently invent answers when the context doesn't contain one.


In [74]:
# ── Generation: the answering prompt ────────────────────────────────────────
# {context} is a placeholder filled with the retrieved chunks at question time.
# This is the heart of RAG: the model is told to answer FROM the supplied
# extracts and to admit when it doesn't know — which curbs hallucination.
SYSTEM_PROMPT = """ 
You are a knowledgeable, helpful assistant representing the company Insurellm.
You are chatting with a user about Insurellm. Your answer will be evaluated for
accuracy, relevance and completeness, so make sure it only answers the question
and fully answers it. If you don't know the answer, say so. For context, here are
specific extracts from the Knowledge Base that might be directly relevant to the 
user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [75]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    # Build the final chat payload:
    #   [system prompt stuffed with retrieved context] + [conversation history] + [user's question]
    # ⚠️ Mind the argument order (question, history, chunks) — calling it in the
    # wrong order either crashes or silently builds a garbage prompt.
    # Each extract is labelled with its source file so the model can cite it.
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [
        {
            'role': 'system',
            'content': system_prompt
        }] + history + [
        {
            'role': 'user',
            'content': question
        }]

In [76]:
# ── Query rewriting ─────────────────────────────────────────────────────────
# Users ask vague, conversational questions ('what about her education?').
# Searching the vector DB with the raw question retrieves poorly, so an LLM
# first rewrites it — using the conversation history — into a short, specific,
# self-contained search query.
def rewrite_query(question, history=None):
    """ Rewrite the user's question to be a more specific question that is more likely to surface relevant content
    in the Knowledge Base."""
    history = history or []   # avoid the mutable-default-argument pitfall
    message = f""" You are in a conversation with a user, answering questions about the company Insurellm.
    You are about to look up information in a Knowledge Base to answer the user's question.
    
    This is the history of your conversation so far with the user:
     {history}

    And this is the user's current question:
     {question}

    Respond only with a single, refined question that you will use to search the Knowledge Base.
    It should be a VERY short specific question most likely to surface content. Focus on the question details.
    Don't mention the company name unless it's a general question about the company.
    IMPORTANT: Respond ONLY with the knowledge base query, nothing else.
    """

    response = completion(model=MODEL,
                          extra_body= {'reasoning_effort': 'none'},
                          messages=[
                              {
                                  'role': 'system',
                                  'content': message
                              }
                          ])
    return response.choices[0].message.content

In [77]:
# Quick test — should come back as a crisp query like 'Who won the IIOTY award?'
rewrite_query("who won the IIOTY award:" , [])

'Who won the IIOTY award?'

## 🔗 The full pipeline — `answer_question()`

Everything composed into four steps:

| Step | Function | What it does |
|---|---|---|
| 1 | `rewrite_query` | turn the (possibly vague) question + history into a sharp search query |
| 2 | `fetch_context` | embed the query → top-K vector search → LLM rerank |
| 3 | `make_rag_messages` | stuff the chunks into the system prompt (answering the **original** question) |
| 4 | `completion` | generate the grounded answer |

Note the subtlety: **search** uses the *rewritten* query, but the model **answers** the *original* question. The retrieved chunks are returned too, so a UI (e.g. Gradio) could display sources.


In [78]:
# ── The complete RAG pipeline in one function ────────────────────────────────
def answer_question(question: str, history: list[dict] | None = None)-> tuple[str, list]:
    """ Answer the question using RAG and return the answer and the retrieved content. """
    history = history or []   # avoid the mutable-default-argument pitfall
    query = rewrite_query(question, history)   # 1) rewrite into a focused KB search query
    print(query)                               #    (debug print — delete for production)
    chunks = fetch_context(query)              # 2) vector search + LLM rerank
    messages = make_rag_messages(question, history, chunks)   # 3) stuff chunks into the prompt
    response = completion(model=MODEL,                        #    (but answer the ORIGINAL question)
                          extra_body={'reasoning_effort': 'none'},
                          messages=messages)   # 4) generate the grounded answer
    return response.choices[0].message.content,chunks   # chunks returned too, so a UI can show sources

In [79]:
from IPython.display import Markdown, display

def show_answer(question, history=None):
    answer, chunks = answer_question(question, history)
    sources = sorted({chunk.metadata['source'] for chunk in chunks})
    display(Markdown(f"### ❓ {question}\n\n{answer}"))
    display(Markdown("**📄 Sources consulted:**\n" + "\n".join(f"- `{s}`" for s in sources)))

In [80]:
# 🎉 End-to-end smoke test: returns (answer, chunks_used)
answer_question('Who won the IIOTY award?', [])

Who won the IIOTY award?
[10, 1, 8, 3, 7, 2, 5, 6, 9, 4]


('Maxine Thompson won Insurellm Innovator of the Year (IIOTY) in 2023.',
 [Result(page_content="Maxine Thompson: Career Progression and Performance (2017–2019)\n\nMaxine progressed from Junior Data Engineer to Data Engineer, working on ETL, data integration, data models, and data quality. Her performance ratings were Meets Expectations in 2017, Exceeds Expectations in 2018, and Needs Improvement in 2019, when personal circumstances affected deadlines and stakeholder communication.\n\n## Insurellm Career Progression\n- **January 2017 - October 2018**: **Junior Data Engineer**  \n  * Maxine joined Insurellm as a Junior Data Engineer, focusing primarily on ETL processes and data integration tasks. She quickly learned Insurellm's data architecture, collaborating with other team members to streamline data workflows.  \n- **November 2018 - December 2020**: **Data Engineer**  \n  * In her new role, Maxine expanded her responsibilities to include designing comprehensive data models and improvi

## ✨ Clean, readable output

`answer_question` returns a raw `(answer, chunks)` tuple, so Jupyter prints it as one
wall of escaped text. Two tricks fix that:

1. **Unpack the tuple** (`answer, chunks = ...`) so the repr is never displayed.
2. **Render the answer as Markdown** with `IPython.display` — headings, bullets and
   bold from the model display properly — and reduce the chunks to a short,
   de-duplicated source list.

This changes *presentation only* — `answer_question` itself is untouched, so it still
returns the raw data a UI (e.g. Gradio) would need.


In [81]:
# Same pipeline, pretty output
show_answer('Who won the IIOTY award?')

Who won the IIOTY award?
[10, 1, 8, 3, 7, 2, 5, 6, 9, 4]


### ❓ Who won the IIOTY award?

Maxine Thompson won the Insurellm Innovator of the Year (IIOTY) award in 2023.

**📄 Sources consulted:**
- `knowledge-base/contracts/Contract with SafeHaven Property Insurance for Homellm.md`
- `knowledge-base/employees/Alex Harper.md`
- `knowledge-base/employees/Avery Lancaster.md`
- `knowledge-base/employees/Daniel Park.md`
- `knowledge-base/employees/James Wilson.md`
- `knowledge-base/employees/Jordan K. Bishop.md`
- `knowledge-base/employees/Maxine Thompson.md`
- `knowledge-base/employees/Michael O'Brien.md`

In [82]:
answer_question("Who is Avery?", [])

Who is Avery at Insurellm?
[1, 2, 3, 4, 7, 8, 5, 6, 10, 9]


('Avery Lancaster is Insurellm’s co-founder and CEO. She founded the company in 2015 and is based in San Francisco. Before Insurellm, she worked in insurance product management and business analysis.',
 [Result(page_content='Avery Lancaster — Profile and Current Role\n\nAvery Lancaster is Insurellm’s Co-Founder and CEO, based in San Francisco, California. Her profile lists her date of birth as March 15, 1985, and current salary as $225,000.\n\n# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  ', metadata={'type': 'employees', 'source': 'knowledge-base/employees/Avery Lancaster.md'}),
  Result(page_content='Career Progression at Insurellm and Prior Roles\n\nAvery co-founded Insurellm in 2015 and has served as CEO, guiding the company’s growth as an Insurance Tech provider. Before that, she was a Senior Product Manager at Innovate Ins

In [83]:
from IPython.display import Markdown, display

def show_answer(question, history=None):
    """ Run the RAG pipeline and render the result nicely in the notebook. """
    answer, chunks = answer_question(question, history)   # unpacking avoids the raw tuple repr
    sources = sorted({chunk.metadata['source'] for chunk in chunks})   # de-duplicated source files
    display(Markdown(f"### ❓ {question}\n\n{answer}"))
    display(Markdown("**📄 Sources consulted:**\n" + "\n".join(f"- `{s}`" for s in sources)))

In [84]:
show_answer("Who is avery")

Who is Avery at Insurellm?
[1, 2, 3, 4, 8, 7, 5, 6, 10, 9]


### ❓ Who is avery

Avery Lancaster is Insurellm’s co-founder and CEO. She founded the company in 2015 and is based in San Francisco.

**📄 Sources consulted:**
- `knowledge-base/company/about.md`
- `knowledge-base/company/careers.md`
- `knowledge-base/company/overview.md`
- `knowledge-base/employees/Avery Lancaster.md`

# 🏁 Conclusion

## What was built

A complete, framework-free RAG system over 76 company documents:

```
INGESTION:  fetch_documents() → create_chunks() → create_embeddings()
                 76 docs            593 chunks        Chroma vector DB

ANSWERING:  answer_question()
              = rewrite_query() → fetch_context_unranked() → rerank()
                → make_rag_messages() → completion()
```

## Key takeaways

1. **RAG = search + prompt stuffing.** No fine-tuning; the model stays frozen while the knowledge lives in a database you can update any time.
2. **Chunking quality drives answer quality.** LLM-chosen boundaries + overlap + headline/summary enrichment beat naive fixed-size splitting.
3. **The embedding model must match** between ingestion and querying — vectors from different models are not comparable.
4. **Retrieve wide, rerank narrow.** Vector search recalls candidates; an LLM reranker restores precision (the Manchester test: buried in 20 results → index 0).
5. **Query rewriting** bridges the gap between how users talk and how knowledge bases are searched — essential once conversation history exists.
6. **Structured outputs (Pydantic `response_format`)** turn LLM replies from "hopefully parseable text" into guaranteed, validated JSON.

## 🐞 Bugs found & fixed while building (learn from these!)

| Bug | Symptom | Lesson |
|---|---|---|
| `return chunks` indented *inside* the retrieval loop | every search silently returned **1** chunk instead of 10/20, making the rerank demos meaningless | an early `return` in a loop is a silent logic bug — outputs looked plausible, just wrong |
| `make_rag_messages(chunks, question)` — wrong argument order | `TypeError: missing 1 required positional argument` | positional args in the wrong order either crash (lucky) or silently build a garbage prompt (unlucky) |

## ✅ Review notes — all implemented

These issues were found during review and have been fixed in the code above:

1. **`RETRIEVAL_K` global mutation** → `fetch_context_unranked(question, k=RETRIEVAL_K)` takes a parameter; the Manchester test passes `k=20` without touching the global. *Impact: `answer_question` now always retrieves 10 chunks (predictable, cheaper). Raise `RETRIEVAL_K` if answers ever lack context.*
2. **Typos in the rerank prompt** ("chunks fo text", "in the qestion") → fixed. *Impact: clearer instructions to the reranker; same behavior.*
3. **`documents` shadowing** in the visualization cells → renamed to `chunk_texts`. *Impact: none at runtime; safer out-of-order re-runs.*
4. **Mutable default `history=[]`** → `history=None` + `history = history or []`. *Impact: none today; removes a classic Python footgun.*
5. **`rerank` trusted the model's ids** → invalid/duplicate ids are dropped and forgotten ids re-appended, so it can neither crash nor lose chunks. *Impact: identical when the model behaves; graceful when it doesn't.*
6. **One giant embedding call** → batched (`EMBED_BATCH_SIZE = 500`). *Impact: same vectors; scales past OpenAI's per-request limits.*
7. **Debug `print`s** (rewritten query, rerank order) — *kept deliberately* in this reference notebook; delete the two marked lines for production.

## 🚀 Where to go next

- **Gradio chat UI** — wire `answer_question` into a chat interface (the course's `app.py` does exactly this).
- **Evaluation** — the `week5/evaluation` folder shows how to score accuracy/relevance/completeness systematically instead of eyeballing.
- **Hybrid search** — combine vector search with keyword (BM25) search for names/acronyms that embeddings miss.
- **Incremental ingestion** — re-chunk only changed files instead of wiping the collection.
- **Citations in answers** — the source metadata is already in the prompt; ask the model to quote its sources explicitly.
